# Document Processing & Chunking
### Practice Notebook

**Assumed pre-installed libraries:** `numpy`, `scikit-learn`, `nltk`

If `nltk`'s sentence tokenizer data isn't downloaded yet, run:
```python
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')
```


In [1]:
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Abhiska\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Abhiska\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

## 1. Why chunking is needed

Documents are almost always too long to embed or feed to an LLM as a single
unit:
- Embedding models have a **max input length** (often 256–8192 tokens).
- Retrieval works best over **small, focused** pieces of text — embedding an
  entire 50-page PDF as one vector blurs together many unrelated topics into
  one average, hurting retrieval precision.
- Smaller chunks let you retrieve *only* the passage relevant to a question,
  keeping the LLM's context window compact and cost-efficient.

So before we can embed or index anything (Days 3–4), we need to split raw
documents into chunks. Let's work with one running example document.


In [2]:
sample_document = """
Retrieval-Augmented Generation (RAG) combines a retrieval system with a
generative language model. Instead of relying solely on knowledge baked into
the model's weights during training, RAG systems fetch relevant text from an
external knowledge base at query time and pass it to the model as context.

This approach has several advantages. First, it allows the system to stay
up to date without retraining the underlying model -- you simply update the
knowledge base. Second, it reduces hallucination, because the model is
encouraged to ground its answer in retrieved text rather than inventing
facts. Third, it enables citation: the system can point back to the exact
source document a claim came from.

However, RAG is not free of challenges. Retrieval quality is a bottleneck --
if the retriever fails to find the relevant chunk, the generator has nothing
good to work with, no matter how capable it is. Chunking strategy, embedding
model choice, and index design all materially affect end-to-end quality.
""".strip()

print(f"Document length: {len(sample_document)} characters, "
      f"~{len(sample_document.split())} words")


Document length: 1008 characters, ~160 words


## 2. Fixed-size chunking

The simplest strategy: split every `N` characters (or tokens), regardless of
sentence or paragraph boundaries.


In [3]:
def fixed_size_chunk(text: str, chunk_size: int = 200, overlap: int = 0) -> list:
    """Split text into fixed-size character chunks with optional overlap."""
    chunks = []
    start = 0
    step = chunk_size - overlap
    if step <= 0:
        raise ValueError("overlap must be smaller than chunk_size")
    while start < len(text):
        chunk = text[start:start + chunk_size]
        chunks.append(chunk)
        start += step
    return chunks

fixed_chunks = fixed_size_chunk(sample_document, chunk_size=200, overlap=0)
for i, c in enumerate(fixed_chunks):
    print(f"--- chunk {i} ({len(c)} chars) ---")
    print(c, "\n")


--- chunk 0 (200 chars) ---
Retrieval-Augmented Generation (RAG) combines a retrieval system with a
generative language model. Instead of relying solely on knowledge baked into
the model's weights during training, RAG systems fe 

--- chunk 1 (200 chars) ---
tch relevant text from an
external knowledge base at query time and pass it to the model as context.

This approach has several advantages. First, it allows the system to stay
up to date without retra 

--- chunk 2 (200 chars) ---
ining the underlying model -- you simply update the
knowledge base. Second, it reduces hallucination, because the model is
encouraged to ground its answer in retrieved text rather than inventing
facts 

--- chunk 3 (200 chars) ---
. Third, it enables citation: the system can point back to the exact
source document a claim came from.

However, RAG is not free of challenges. Retrieval quality is a bottleneck --
if the retriever f 

--- chunk 4 (200 chars) ---
ails to find the relevant chunk, the generator h

Notice how fixed-size chunking can cut a sentence — or even a word — right
down the middle. That's the trade-off: simplicity and predictable size, at
the cost of chunk *coherence*.


## 3. Recursive chunking

A smarter approach: try to split on the "biggest" natural boundary first
(paragraphs), and only fall back to smaller boundaries (sentences, then
words, then characters) if a piece is still too big. This keeps chunks
semantically coherent while still respecting a max size.


In [5]:
def recursive_chunk(text: str, chunk_size: int = 300,
                     separators=None) -> list:
    """A simplified version of the 'recursive character splitter' idea
    popularized by LangChain: try each separator in order, from coarsest
    (paragraph) to finest (character), splitting only where needed.
    """
    if separators is None:
        separators = ["\n\n", "\n", ". ", " ", ""]

    def _split(text, seps):
        if len(text) <= chunk_size:
            return [text] if text.strip() else []
        if not seps:
            # last resort: hard character split
            return [text[i:i + chunk_size] for i in range(0, len(text), chunk_size)]
        sep = seps[0]
        parts = text.split(sep) if sep else list(text)
        # Greedily merge parts back together up to chunk_size
        chunks, current = [], ""
        for part in parts:
            candidate = current + sep + part if current else part
            if len(candidate) <= chunk_size:
                current = candidate
            else:
                if current:
                    chunks.extend(_split(current, seps[1:]))
                current = part
        if current:
            chunks.extend(_split(current, seps[1:]))
        return chunks

    return [c.strip() for c in _split(text, separators) if c.strip()]

recursive_chunks = recursive_chunk(sample_document, chunk_size=300)
for i, c in enumerate(recursive_chunks):
    print(f"--- chunk {i} ({len(c)} chars) ---")
    print(c, "\n")


--- chunk 0 (300 chars) ---
Retrieval-Augmented Generation (RAG) combines a retrieval system with a
generative language model. Instead of relying solely on knowledge baked into
the model's weights during training, RAG systems fetch relevant text from an
external knowledge base at query time and pass it to the model as context. 

--- chunk 1 (292 chars) ---
This approach has several advantages. First, it allows the system to stay
up to date without retraining the underlying model -- you simply update the
knowledge base. Second, it reduces hallucination, because the model is
encouraged to ground its answer in retrieved text rather than inventing 

--- chunk 2 (108 chars) ---
facts. Third, it enables citation: the system can point back to the exact
source document a claim came from. 

--- chunk 3 (230 chars) ---
However, RAG is not free of challenges. Retrieval quality is a bottleneck --
if the retriever fails to find the relevant chunk, the generator has nothing
good to work with, no mat

**Note for production use:** LangChain's `RecursiveCharacterTextSplitter` and
LlamaIndex's `SentenceSplitter` implement this same idea with more edge-case
handling. The function above shows you what's happening *under the hood*.


## 4. Sentence-based chunking

Split strictly on sentence boundaries, then group sentences together until
adding one more would exceed the target size. This guarantees you never cut
a sentence in half.


In [5]:
import nltk
# Uncomment on first run:
# nltk.download('punkt')
# nltk.download('punkt_tab')
from nltk.tokenize import sent_tokenize

def sentence_chunk(text: str, max_chunk_size: int = 300) -> list:
    sentences = sent_tokenize(text)
    chunks, current = [], ""
    for sent in sentences:
        candidate = f"{current} {sent}".strip()
        if len(candidate) <= max_chunk_size:
            current = candidate
        else:
            if current:
                chunks.append(current)
            current = sent
    if current:
        chunks.append(current)
    return chunks

sentence_chunks = sentence_chunk(sample_document, max_chunk_size=300)
for i, c in enumerate(sentence_chunks):
    print(f"--- chunk {i} ({len(c)} chars) ---")
    print(c, "\n")


--- chunk 0 (300 chars) ---
Retrieval-Augmented Generation (RAG) combines a retrieval system with a
generative language model. Instead of relying solely on knowledge baked into
the model's weights during training, RAG systems fetch relevant text from an
external knowledge base at query time and pass it to the model as context. 

--- chunk 1 (299 chars) ---
This approach has several advantages. First, it allows the system to stay
up to date without retraining the underlying model -- you simply update the
knowledge base. Second, it reduces hallucination, because the model is
encouraged to ground its answer in retrieved text rather than inventing
facts. 

--- chunk 2 (141 chars) ---
Third, it enables citation: the system can point back to the exact
source document a claim came from. However, RAG is not free of challenges. 

--- chunk 3 (263 chars) ---
Retrieval quality is a bottleneck --
if the retriever fails to find the relevant chunk, the generator has nothing
good to work with, no mat

## 5. Semantic chunking (concept preview)

Semantic chunking goes one step further: instead of a fixed size, it groups
*sentences whose embeddings are similar* into the same chunk, and starts a
new chunk when the topic shifts (detected as a drop in embedding similarity
between consecutive sentences). We'll implement this for real on Day 3 once
embeddings are introduced — for now, here's the pseudocode so the concept is
in place before then:

```
sentences = split_into_sentences(document)
embeddings = [embed(s) for s in sentences]

chunks = []
current_chunk = [sentences[0]]
for i in range(1, len(sentences)):
    similarity = cosine_similarity(embeddings[i-1], embeddings[i])
    if similarity < SIMILARITY_THRESHOLD:
        chunks.append(current_chunk)      # topic shift -> close this chunk
        current_chunk = [sentences[i]]
    else:
        current_chunk.append(sentences[i])
chunks.append(current_chunk)
```

**Exercise 2.1 (come back to this after Day 3):** implement this function for
real using `sentence-transformers` embeddings and `sklearn`'s
`cosine_similarity`, and compare the resulting chunk boundaries against the
fixed-size and recursive chunks above.


In [8]:
my_text = """
Artificial Intelligence is transforming medical diagnostics by analyzing complex imaging data. 
Machine learning models can detect subtle anomalies in X-rays and MRIs faster than human inspection.
However, integrating these models into clinical workflows requires careful validation. 
Data privacy, algorithmic bias, and regulatory approvals remain major hurdles before widespread adoption.
""".strip()

fixed_chunks = fixed_size_chunk(my_text, chunk_size=150, overlap=0)
records_fixed = build_chunk_records(fixed_chunks, source="ai_health.txt")

sentence_chunks = sentence_chunk(my_text, max_chunk_size=150)
records_sentence = build_chunk_records(sentence_chunks, source="ai_health.txt")

print("=== Strategy A: Fixed-Size Chunk Records ===")
for r in records_fixed:
    print(r)

print("\n=== Strategy B: Sentence-Based Chunk Records ===")
for r in records_sentence:
    print(r)

=== Strategy A: Fixed-Size Chunk Records ===
{'chunk_id': 'ai_health.txt::chunk_0', 'source': 'ai_health.txt', 'chunk_index': 0, 'char_span': (0, 150), 'text': 'Artificial Intelligence is transforming medical diagnostics by analyzing complex imaging data. \nMachine learning models can detect subtle anomalies in'}
{'chunk_id': 'ai_health.txt::chunk_1', 'source': 'ai_health.txt', 'chunk_index': 1, 'char_span': (150, 300), 'text': ' X-rays and MRIs faster than human inspection.\nHowever, integrating these models into clinical workflows requires careful validation. \nData privacy, a'}
{'chunk_id': 'ai_health.txt::chunk_2', 'source': 'ai_health.txt', 'chunk_index': 2, 'char_span': (300, 390), 'text': 'lgorithmic bias, and regulatory approvals remain major hurdles before widespread adoption.'}

=== Strategy B: Sentence-Based Chunk Records ===
{'chunk_id': 'ai_health.txt::chunk_0', 'source': 'ai_health.txt', 'chunk_index': 0, 'char_span': (0, 94), 'text': 'Artificial Intelligence is transform

## 6. Chunk overlap and size trade-offs

| Chunk size | Pros | Cons |
|---|---|---|
| **Small** (e.g., 100–200 chars) | Precise retrieval, less irrelevant text per chunk | Loses surrounding context; may need to retrieve many chunks |
| **Large** (e.g., 1000+ chars) | More context per chunk, fewer retrievals needed | Embedding blurs multiple ideas together; retrieval precision drops |

**Overlap** (e.g., repeating the last 20% of one chunk at the start of the
next) helps prevent losing information that straddles a chunk boundary — at
the cost of some storage/index redundancy.




**Exercise 2.2:** re-run `fixed_size_chunk` with `chunk_size=200, overlap=50`
and inspect how consecutive chunks now share text. When would that overlap matter for retrieval quality?

When overlap matters:-Prevents losing context when key terms cross chunk boundries.

 Give a concrete example sentence that would be
split awkwardly without it.

**Example:**

- **Without overlap:**
  - Chunk 0 ends with: "..RAG systems fe"
  - Chunk 1 starts with: "tch relevant text..."
  (The word "fetch" gets broken in half mid)

- **With 50-character overlap:**
  - Chunk 1 includes: "RAG systems fetch relevant text"
  (The entire phrase stays completely allowing the vector retriever)



In [9]:
overlapping_chunks = fixed_size_chunk(sample_document, chunk_size=200, overlap=50)
for i, c in enumerate(overlapping_chunks[:3]):
    print(f"--- chunk {i} ---")
    print(c, "\n")


--- chunk 0 ---
Retrieval-Augmented Generation (RAG) combines a retrieval system with a
generative language model. Instead of relying solely on knowledge baked into
the model's weights during training, RAG systems fe 

--- chunk 1 ---
he model's weights during training, RAG systems fetch relevant text from an
external knowledge base at query time and pass it to the model as context.

This approach has several advantages. First, it  

--- chunk 2 ---


This approach has several advantages. First, it allows the system to stay
up to date without retraining the underlying model -- you simply update the
knowledge base. Second, it reduces hallucination 



## 7. Metadata and document preprocessing basics

In a real pipeline, every chunk should carry metadata alongside its text —
this is what lets you filter search results (e.g., "only search documents
from 2025") and cite sources back to the user.


In [7]:
def build_chunk_records(chunks: list, source: str) -> list:
    """Attach metadata to each chunk: a stable id, source document name,
    chunk index, and character span (useful for citing back to the original
    document).
    """
    records = []
    cursor = 0
    for i, chunk in enumerate(chunks):
        start = cursor
        end = start + len(chunk)
        records.append({
            "chunk_id": f"{source}::chunk_{i}",
            "source": source,
            "chunk_index": i,
            "char_span": (start, end),
            "text": chunk,
        })
        cursor = end
    return records

records = build_chunk_records(sentence_chunks, source="rag_intro_doc.txt")
for r in records[:2]:
    print(r)


{'chunk_id': 'rag_intro_doc.txt::chunk_0', 'source': 'rag_intro_doc.txt', 'chunk_index': 0, 'char_span': (0, 300), 'text': "Retrieval-Augmented Generation (RAG) combines a retrieval system with a\ngenerative language model. Instead of relying solely on knowledge baked into\nthe model's weights during training, RAG systems fetch relevant text from an\nexternal knowledge base at query time and pass it to the model as context."}
{'chunk_id': 'rag_intro_doc.txt::chunk_1', 'source': 'rag_intro_doc.txt', 'chunk_index': 1, 'char_span': (300, 599), 'text': 'This approach has several advantages. First, it allows the system to stay\nup to date without retraining the underlying model -- you simply update the\nknowledge base. Second, it reduces hallucination, because the model is\nencouraged to ground its answer in retrieved text rather than inventing\nfacts.'}


**Preprocessing basics worth doing before chunking** (brief, for discussion):
- Strip boilerplate (headers/footers, nav menus in scraped HTML)
- Normalize whitespace and encoding
- Preserve structure that matters for meaning (e.g., don't discard table
  structure by flattening it to plain text without care)
- Keep a pointer back to the original document/page so retrieved chunks can
  be cited

**Exercise 2.3 (mini deliverable):** Take a paragraph of your own choosing
(a news article, a textbook excerpt, anything ~1–2 paragraphs long), and
produce chunk records using **two different strategies** from this notebook
(e.g., fixed-size vs. sentence-based). Compare: which strategy preserved
meaning better? Which would you pick for a real RAG system, and why?


- **Meaning Preserved:**—it keeps full sentences instead of splitting words mid-thought.
- **RAG Choice:**—vector models and LLMs need complete, context to prevent search misses and hallucinated answers